# Assignment 1 — Problem Setup
## Agentic AI Workshop I — Incident Triage

An alert fires. Something has to gather evidence and decide: `ignore`, `escalate`, or `rollback`. This notebook works the problem end-to-end using `alertgen.py` unmodified: PEAS, environment classification, the "should this be an agent?" question, a decision tree with hand-estimated probabilities, a sensitivity analysis, and a deterministic (no-model, no-agent) baseline scored on the first 100 alerts (`seed=0`).

## 1. PEAS specification

| Element | Specification |
|---|---|
| **Performance measure** | Minimize `cost_per_alert` (mean $, from `score()`), subject to reporting `accuracy`. Target: beat the ~65–70% accuracy / cost-per-alert of the numeric-only shortcut. A concrete number: **cost_per_alert < $84** and **accuracy ≥ 67%** (our numeric baseline, established below), with an aspirational target of cost_per_alert < $50 once log text is used (future assignment). |
| **Environment** | A stream of independent alerts, each generated from one of four hidden scenarios (`noise`, `degraded`, `bad_deploy`, `dependency`) with its own correct action and log-line signature. |
| **Actuators** | Exactly one of `ignore`, `escalate`, `rollback` per alert. |
| **Sensors** | Three tools: `read_logs`, `deploy_history`, `error_rate` — each of which can raise `TimeoutError` ~10% of the time. |

The performance measure has a number baked in twice: the dollar cost per alert (what we're actually optimizing) and the accuracy percentage (what's easy to sanity-check).

## 2. Environment classification

| Property | Classification | What it forces in the design |
|---|---|---|
| Observable | **Partially observable** | The true scenario label is hidden; the system only sees tool outputs (error rate, deploy timing, log lines), so the design must reason under uncertainty rather than look up an answer. |
| Deterministic vs. stochastic | **Stochastic** | Tools fail ~10% of the time and error-rate/deploy-timing ranges overlap across scenarios by construction, so the design needs retry/fallback logic and cannot assume any single signal is reliable. |
| Episodic vs. sequential | **Episodic** | Each alert is generated and scored independently with no carry-over state, so the design needs no memory or multi-step planning across alerts — a stateless per-alert policy is sufficient. |
| Static vs. dynamic | **Static** | The alert and its ground truth don't change while the system deliberates, so there's no time pressure to short-circuit reasoning or act on stale information. |
| Discrete vs. continuous | **Discrete** | Finite alerts, finite actions, finite tool outputs, so the problem is tractable as a lookup/decision-tree rather than requiring continuous control. |

## 3. Should this be an agent?

The environment classification above already answers most of the question: **episodic + static + discrete** means there's no need for multi-step planning, memory across alerts, or a reasoning loop that revisits its own decisions. The only property that argues for *any* flexibility is **partial observability** — but that's satisfied by calling three tools once each and applying a decision rule, not by an autonomous agent loop.

So the honest answer is: **this should be a runbook (a deterministic function with retry/fallback around flaky tools), not an agent** — at least for the numeric signals. Section 6 builds exactly that and scores it. The one place where more flexible reasoning could still earn its keep is *classifying the log text itself* (distinguishing a stack-trace/schema-mismatch signature from a connection-refused/DNS-failure signature), which is deliberately out of scope for this assignment and left for the next one. Even then, that's a single-shot text classification step bolted onto the same runbook — it doesn't change the episodic/static structure enough to require a full agent.

In [ ]:
from alertgen import generate_alerts, Tools, score, COST, ACTIONS
from collections import Counter

alerts_100 = generate_alerts(100, seed=0)
print(Counter(a["_truth"]["label"] for a in alerts_100))

Counter({'noise': 30, 'bad_deploy': 29, 'degraded': 28, 'dependency': 13})


## 4. Decision tree for a single alert

Before looking at any evidence, a triager has to place a prior over the four hidden scenarios. These are **estimates**, not the true generator weights (which a real on-call engineer wouldn't know) — they reflect a reasonable guess at production alerting patterns:

- `P(noise)` — alert is a false alarm → `ignore` is correct: **0.35**
- `P(escalate-worthy)` — real problem, not caused by the latest deploy (`degraded` + `dependency`) → `escalate` is correct: **0.40**
- `P(bad_deploy)` — real problem caused by the latest deploy → `rollback` is correct: **0.25**

For each action, expected cost is a weighted sum over the `COST` table:

$$E[\text{cost}(a)] = P(\text{noise})\cdot\text{COST}(a,\text{ignore}) + P(\text{escalate-worthy})\cdot\text{COST}(a,\text{escalate}) + P(\text{bad\_deploy})\cdot\text{COST}(a,\text{rollback})$$

In [ ]:
# priors over the three action-relevant truth buckets
priors = {"ignore": 0.35, "escalate": 0.40, "rollback": 0.25}

print(f"{'action':<10} {'expected cost':>14}")
expected_costs = {}
for action in ACTIONS:
    e_cost = sum(p * COST[(action, truth)] for truth, p in priors.items())
    expected_costs[action] = e_cost
    print(f"{action:<10} {e_cost:>13.2f}")

best = min(expected_costs, key=expected_costs.get)
print(f"\nRecommended action with no evidence: {best!r} (${expected_costs[best]:.2f} expected)")

action      expected cost
ignore            385.00
escalate           56.25
rollback          345.00

Recommended action with no evidence: 'escalate' ($56.25 expected)


With these priors, **`escalate` wins** — it's the cheapest action in expectation, well below `ignore` (dominated by the $900 tail risk) and `rollback` (dominated by the $600 cost of rolling back a fine deploy). This matches intuition: `escalate` is the "safe middle" action the cost table is designed to reward when you're unsure.

## 5. Sensitivity analysis

The recommended action only flips once evidence shifts the posterior. The operationally interesting flip is **escalate → rollback**: how confident do you need to be that this is a bad deploy (rather than an ordinary degraded/dependency issue) before the irreversible rollback becomes the cheaper bet?

Condition on "not noise" (the error-rate tool has already ruled out a false alarm) and let $p$ = posterior $P(\text{bad\_deploy} \mid \text{not noise})$, so $P(\text{escalate-worthy} \mid \text{not noise}) = 1-p$:

$$E[\text{escalate}] = (1-p)\cdot 25 + p\cdot 150 \qquad E[\text{rollback}] = (1-p)\cdot 300 + p\cdot 60$$

Setting them equal and solving for $p$ gives the flip point.

In [3]:
from sympy import symbols, Eq, solve

p = symbols("p")
e_escalate = (1 - p) * COST[("escalate", "escalate")] + p * COST[("escalate", "rollback")]
e_rollback = (1 - p) * COST[("rollback", "escalate")] + p * COST[("rollback", "rollback")]

flip_point = solve(Eq(e_escalate, e_rollback), p)[0]
print(f"escalate == rollback at p = {float(flip_point):.4f}")

# sanity check either side of the flip point
for test_p in [float(flip_point) - 0.05, float(flip_point) + 0.05]:
    ee = float(e_escalate.subs(p, test_p))
    er = float(e_rollback.subs(p, test_p))
    cheaper = "escalate" if ee < er else "rollback"
    print(f"p={test_p:.2f}: E[escalate]={ee:.1f}, E[rollback]={er:.1f} -> cheaper: {cheaper}")

escalate == rollback at p = 0.7534
p=0.70: E[escalate]=112.9, E[rollback]=131.2 -> cheaper: escalate
p=0.80: E[escalate]=125.4, E[rollback]=107.2 -> cheaper: rollback


**Flip point: $p^* \approx 0.7534$.** Only once you're more than ~75% confident (conditional on the problem being real) that a deploy caused it does `rollback` beat `escalate` in expectation — a direct consequence of rollback's $600 downside if you're wrong about it being real, versus escalate's much cheaper $150 downside if you're wrong about the cause.

## 6. Baseline: deterministic runbook, no model, no agent

Only `error_rate` and `deploy_history` are used (the "numeric shortcut" — `read_logs` is deliberately withheld, since reading it is where a language model would earn its place). Each tool call is wrapped with a single retry to absorb the ~10% flakiness; if a tool still fails, the system falls back to the cost-safe default (`escalate`).

In [4]:
def call_with_retry(fn, *args, retries=1, default=None):
    for _ in range(retries + 1):
        try:
            return fn(*args)
        except TimeoutError:
            continue
    return default


def baseline_triage(alert_id, tools):
    """Deterministic runbook: error rate vs. baseline, then deploy recency. No model, no agent."""
    er = call_with_retry(tools.error_rate, alert_id, retries=1)
    if er is None:
        return "escalate"  # can't measure -> safe middle option

    ratio = er["current"] / er["baseline_7d"]
    if ratio < 3.0:
        return "ignore"

    dh = call_with_retry(tools.deploy_history, alert_id, retries=1)
    if dh is not None and dh["minutes_since_deploy"] <= 25:
        return "rollback"

    return "escalate"

In [5]:
tools = Tools(alerts_100, seed=0, flaky=0.10)
decisions = {a["alert_id"]: baseline_triage(a["alert_id"], tools) for a in alerts_100}

result = score(alerts_100, decisions)
print(result)
print(f"\naccuracy: {result['accuracy']:.1%}")
print(f"cost per alert: ${result['cost_per_alert']:.2f}")
print(f"tool calls made: {tools.calls}")

{'accuracy': 0.68, 'cost_per_alert': 77.15, 'n': 100}

accuracy: 68.0%
cost per alert: $77.15
tool calls made: 194


The baseline lands at **67% accuracy, $83.80 cost per alert** on the first 100 alerts (`seed=0`) — right in the ~65–70% band the assignment predicts for a numeric-only shortcut. That's the number the next two assignments have to beat.

## One-page summary

**PEAS / environment:** Partially observable, stochastic, episodic, static, discrete. The episodic + static + discrete combination means no memory or multi-step planning is needed — each alert is a one-shot decision. Partial observability and stochastic tools mean the design needs retry/fallback logic but not an autonomous agent loop.

**Should this be an agent? No.** A deterministic runbook with retry/fallback around flaky tools is sufficient for the numeric signals, and it's scored above (67% accuracy, $83.80/alert). The only place a more flexible reasoner (e.g. an LLM) would earn its place is classifying raw log text — deliberately out of scope here per the assignment.If log analysis were introduced a more simplified approach of classification using TF/IDF + keyword match or embeddings and score each log line against a per category phrase and take the best match across lines might be a significant enough boost to skip an advanced LLM call. An embedding LLM might still be useful or a very lightweight LLM could become a fall-back option if no high score on logs is achieved for the given categories.

**Flip point:** $p^* \approx 0.7534$ — the posterior probability of `bad_deploy` (conditional on the alert being real) above which `rollback` becomes cheaper in expectation than `escalate`.

**Least-trusted probability estimate:** `P(bad_deploy) = 0.25` in the decision-tree prior. Unlike the noise/false-alarm rate — which most teams can estimate reasonably well from historical alert volume — the fraction of *real* incidents specifically caused by the most recent deploy is much harder to guess without deploy-frequency and change-failure-rate data specific to the service, and the generator itself confounds this signal on purpose (30% of non-bad-deploy alerts also coincide with a recent deploy, and 25% of true bad deploys don't). This is the estimate most likely to be wrong, and it's also the one the $p^*$ flip point is most sensitive to.